# Molecule CV — Train Models and Save Predictions

Trains the 3 CNN architectures (ADMET / Custom / Toxic Colors) on the BACE
dataset and saves per-molecule predictions, model metrics, and training
histories to CSV. No statistical analysis here — that lives in a separate
local notebook.

**Runtime:** T4 GPU recommended. ~10 min total.

**Outputs (in `results/`):**
- `predictions.csv` — one row per test molecule, columns for actual pIC50, Class, and each model's prediction
- `model_metrics.csv` — test loss, MAE, AUC per model
- `history_<model>.csv` — training history per model

In [ ]:
# Mount Drive if using Colab
from google.colab import drive
drive.mount('/content/drive')

# Set this to wherever the repo/data lives
DATA_DIR = '/content/drive/MyDrive/Molecule_CV'  # adjust as needed

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, mean_absolute_error
import matplotlib.pyplot as plt

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"TF: {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")

## Load Data

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'bace.csv'))
print(f"Samples: {len(df)}, Columns: {len(df.columns)}")
print(f"Class distribution: {df['Class'].value_counts().to_dict()}")
print(f"pIC50 range: {df['pIC50'].min():.2f} — {df['pIC50'].max():.2f}")

In [ ]:
IMG_DIR = os.path.join(DATA_DIR, 'molecule_images')

def load_images_and_targets(df, img_dir, img_size=(180, 180)):
    """Load molecule images and extract targets."""
    images, pic50s, classes, cids = [], [], [], []
    missing = 0
    for _, row in df.iterrows():
        img_path = os.path.join(img_dir, f"{row['CID']}.png")
        if not os.path.exists(img_path):
            missing += 1
            continue
        img = tf.io.read_file(img_path)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32) / 255.0
        images.append(img)
        pic50s.append(row['pIC50'])
        classes.append(row['Class'])
        cids.append(row['CID'])
    if missing:
        print(f"Warning: {missing} images not found")
    return np.array(images), np.array(pic50s), np.array(classes), cids

images, pic50, labels, cids = load_images_and_targets(df, IMG_DIR)
print(f"Loaded {len(images)} images, shape: {images[0].shape}")

In [ ]:
# Same split strategy as existing notebooks: 80/10/10
X_temp, X_test, y_temp, y_test, c_temp, c_test, cid_temp, cid_test = train_test_split(
    images, pic50, labels, cids, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val, c_train, c_val, cid_train, cid_val = train_test_split(
    X_temp, y_temp, c_temp, cid_temp, test_size=0.5, random_state=SEED)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

## Define Architectures

All three from the existing notebooks — non-pretrained.

In [ ]:
def build_admet_model(input_shape=(180, 180, 3)):
    """Shi et al. 2019"""
    return keras.Sequential([
        keras.layers.Conv2D(16, (21, 21), activation='relu', input_shape=input_shape),
        keras.layers.MaxPooling2D((14, 14), padding='same'),
        keras.layers.Flatten(),
        keras.layers.Dense(512, activation='relu'),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(1)
    ])

def build_custom_cnn(input_shape=(180, 180, 3)):
    """Multi-layer CNN (test_model_PIC)"""
    return keras.Sequential([
        keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(64, (3, 3), activation='relu'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(128, (3, 3), activation='relu'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(256, (3, 3), activation='relu'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Flatten(),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(1)
    ])

def build_toxic_colors(input_shape=(180, 180, 3)):
    """Fernandez et al. 2018"""
    return keras.Sequential([
        keras.layers.Conv2D(12, (16, 16), activation='relu', input_shape=input_shape),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.4),
        keras.layers.Flatten(),
        keras.layers.Dense(40, activation='relu'),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(1)
    ])

MODELS = {
    'ADMET (Shi 2019)': build_admet_model,
    'Custom CNN': build_custom_cnn,
    'Toxic Colors (Fernandez 2018)': build_toxic_colors,
}

## Train & Collect Predictions

In [ ]:
results = {}  # model_name -> {history, predictions, metrics}

for name, builder in MODELS.items():
    print(f"\n{'='*50}")
    print(f"Training: {name}")
    print(f"{'='*50}")

    model = builder()
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    epochs = 100 if name != 'Custom CNN' else 50
    batch_size = 128 if name != 'Custom CNN' else 32

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=1
    )

    test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
    preds = model.predict(X_test, verbose=0).flatten()

    results[name] = {
        'history': history.history,
        'predictions': preds,
        'test_loss': test_loss,
        'test_mae': test_mae,
    }
    print(f"\n{name} — Test Loss: {test_loss:.4f}, Test MAE: {test_mae:.4f}")

## Save Results to CSV

In [ ]:
OUTPUT_DIR = os.path.join(DATA_DIR, 'results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 1. Predictions per molecule ---
pred_df = pd.DataFrame({'CID': cid_test, 'pIC50_actual': y_test, 'Class': c_test})
for name, res in results.items():
    col = name.split('(')[0].strip().replace(' ', '_').lower()
    pred_df[f'pred_{col}'] = res['predictions']
pred_df.to_csv(os.path.join(OUTPUT_DIR, 'predictions.csv'), index=False)
print(f"Saved predictions.csv ({len(pred_df)} rows)")

# --- 2. Model summary metrics ---
metrics_rows = []
for name, res in results.items():
    auc = roc_auc_score(c_test, res['predictions'])
    metrics_rows.append({
        'model': name,
        'test_mse': res['test_loss'],
        'test_mae': res['test_mae'],
        'auc_roc': auc,
    })
metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'model_metrics.csv'), index=False)
print("Saved model_metrics.csv")
print(metrics_df.to_string(index=False))

# --- 3. Training history ---
for name, res in results.items():
    col = name.split('(')[0].strip().replace(' ', '_').lower()
    hist_df = pd.DataFrame(res['history'])
    hist_df.index.name = 'epoch'
    hist_df.to_csv(os.path.join(OUTPUT_DIR, f'history_{col}.csv'))
print("Saved training histories")